In [1]:
%gui qt
from receiver import TJReceiver
from mantis.mini_interpreter import RawDataInterpreter
from qtpy import QtCore
import numpy as np
import matplotlib.pyplot as plt

%matplotlib qt

In [2]:
hit_dtype = np.dtype(
    [
        ("col", "<i2"),
        ("row", "<i2"),
        ("timestamp", "<i8"),
        ("rx_id", "<i2"),
    ]
)

received_data = []
total_occupancy = None

In [3]:
def handle_data(data):
    received_data.append(data)


tj_receiver = TJReceiver(frontend="tcp://127.0.0.1:5501")
tj_receiver.worker.data.connect(handle_data)
tj_receiver.start()

In [4]:
def make_hitmap(raw_data, rx_id=0):
    interpreter = RawDataInterpreter(rx_id=rx_id)
    hit_buffer = np.zeros(4 * len(raw_data), dtype=hit_dtype)
    hits = interpreter.interpret(raw_data, hit_buffer)  # noqa: F841
    occupancy = interpreter.hist_occ.sum(axis=2)
    return occupancy

In [5]:
fig, ax = plt.subplots()
img = ax.imshow(np.zeros((100, 100)), cmap="hot", interpolation="nearest")
plt.colorbar(img, ax=ax)
ax.set_title("TJ-Monopix2 Hitmap")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
fig.show()

In [6]:
def update_hitmap(occupancy):
    img.set_data(occupancy)
    img.set_clim(vmin=0, vmax=occupancy.max())
    fig.canvas.draw_idle()

In [7]:
last_processed = -1


def process_latest_packet():
    global total_occupancy, last_processed

    if len(received_data) == 0:
        return

    if last_processed == len(received_data) - 1:
        return

    last_processed = len(received_data) - 1
    packet = received_data[last_processed]
    raw_data = packet["raw"].astype(np.uint32)

    occ = make_hitmap(raw_data)

    if total_occupancy is None:
        total_occupancy = np.zeros_like(occ)

    total_occupancy += occ

    update_hitmap(total_occupancy)

In [8]:
timer = QtCore.QTimer()
timer.timeout.connect(process_latest_packet)
timer.start(500)  # run every x ms